# 🏭 Tensor Surgery — Round 1 | Notebook 2
## Cirrhosis Outcome Prediction

### ⚡ Ultra-Fast Multi-Class Gradient Boosting Ensemble

**Competition goal:** Predict the operating outcome of industrial patient from clinical measurements and patient metadata.

Classes:
- `C` → Censored
- `CL` → Censored / living outcome class
- `D` → Death outcome class

This participant-facing notebook follows the same overall pipeline philosophy as the reference competition notebook:
centralized hyperparameters, validation, domain-inspired feature engineering, stratified K-fold training, three boosting models, probability blending, and Kaggle submission generation.

> **Important:** The target column in this dataset is `Outcome`. The submission columns are `Outcome_S`, `Outcome_W`, and `Outcome_F`.


## 📑 Table of Contents

1. Environment Setup & Imports
2. Dataset Placeholders & Path Configuration
3. Hyperparameter Control Center
4. Data Loading & Integrity Checks
5. Domain-Informed Feature Engineering
6. Stratified K-Fold Training & Ensembling
7. Validation Performance
7. OOF Validation & Probability Ensemble
8. Submission Generation & Integrity Checks


In [1]:
# ==============================================================================
# 📦 SECTION 1: ENVIRONMENT SETUP & IMPORTS
# ==============================================================================
import os
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, log_loss, classification_report, confusion_matrix

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
np.random.seed(42)

print("✅ Libraries successfully loaded and initialized!")


✅ Libraries successfully loaded and initialized!


## 📂 SECTION 2: DATASET PLACEHOLDERS & PATH CONFIGURATION

The competition dataset is attached through Kaggle's **Competitions** input area.

Expected Kaggle path:

`/kaggle/input/competitions/tensor-surgery-round-1-notebook-2/`


In [2]:
# ==============================================================================
# 📂 SECTION 2: DATASET PATH CONFIGURATION
# ==============================================================================

DATASET_DIR = "/kaggle/input/competitions/tensor-surgery-round-1-notebook-2"

TRAIN_PATH = DATASET_DIR + "/train.csv"
TEST_PATH = DATASET_DIR + "/test.csv"
SAMPLE_SUB_PATH = DATASET_DIR + "/sample_submission.csv"

# Kaggle writable output directory
OUTPUT_SUB_PATH = "/kaggle/working/submission.csv"

print("📂 Dataset Paths:")
print(f"   • Train Data:        {TRAIN_PATH}")
print(f"   • Test Data:         {TEST_PATH}")
print(f"   • Sample Submission: {SAMPLE_SUB_PATH}")
print(f"   • Output Submission: {OUTPUT_SUB_PATH}")

# ==============================================================================
# 🔍 VERIFY DATASET FILES
# ==============================================================================

required_files = {
    "Train": TRAIN_PATH,
    "Test": TEST_PATH,
    "Sample Submission": SAMPLE_SUB_PATH
}

for name, path in required_files.items():
    if os.path.isfile(path):
        print(f"   ✅ {name}: Found")
    else:
        print(f"   ❌ {name}: NOT FOUND")

missing_files = [
    name for name, path in required_files.items()
    if not os.path.isfile(path)
]

if missing_files:
    raise FileNotFoundError(
        f"\n❌ Missing files: {missing_files}\n"
        f"Expected dataset directory:\n{DATASET_DIR}"
    )

print("\n✅ All competition files found successfully!")

📂 Dataset Paths:
   • Train Data:        /kaggle/input/competitions/tensor-surgery-round-1-notebook-2/train.csv
   • Test Data:         /kaggle/input/competitions/tensor-surgery-round-1-notebook-2/test.csv
   • Sample Submission: /kaggle/input/competitions/tensor-surgery-round-1-notebook-2/sample_submission.csv
   • Output Submission: /kaggle/working/submission.csv
   ✅ Train: Found
   ✅ Test: Found
   ✅ Sample Submission: Found

✅ All competition files found successfully!


## 🎛️ SECTION 3: HYPERPARAMETERS & CONFIGURATION CONTROL CENTER

Participants should primarily experiment with the parameters below.

| Category | Hyperparameter | Starting Value |
|---|---|---:|
| Validation | `N_SPLITS` | `5` |
| Validation | `RANDOM_STATE` | `42` |
| Ensemble | LGBM weight | `0.45` |
| Ensemble | XGB weight | `0.45` |
| Ensemble | CatBoost weight | `0.10` |
| LightGBM | `n_estimators` | `320` |
| LightGBM | `learning_rate` | `0.035` |
| LightGBM | `max_depth` | `6` |
| LightGBM | `num_leaves` | `31` |
| XGBoost | `n_estimators` | `320` |
| XGBoost | `learning_rate` | `0.035` |
| XGBoost | `max_depth` | `5` |
| CatBoost | `iterations` | `300` |
| CatBoost | `depth` | `5` |

The feature engineering and data loading cells should remain compatible with the supplied competition files.


In [3]:
# ==============================================================================
# 🎛️ SECTION 3: HYPERPARAMETER CONFIGURATION
# ==============================================================================

CONFIG = {
    "N_SPLITS": 5,
    "RANDOM_STATE": 42,

    "USE_LGBM": True,
    "USE_XGB": True,
    "USE_CATBOOST": True,

    "WEIGHTS": {
        "lgbm": 0.325,
        "xgb": 0.575,
        "catboost": 0.100
    },

    "LGBM_PARAMS": {
        "n_estimators": 500,
        "learning_rate": 0.025,
        "max_depth": 7,
        "num_leaves": 25,
        "subsample": 0.85,
        "colsample_bytree": 0.75,
        "min_child_samples": 40,
        "random_state": 42,
        "verbosity": -1,
        "n_jobs": 4
    },

    "XGB_PARAMS": {
        "n_estimators": 600,
        "learning_rate": 0.025,
        "max_depth": 4,
        "subsample": 0.85,
        "colsample_bytree": 0.10,
        "min_child_weight": 2,
        "tree_method": "hist",
        "random_state": 42,
        "eval_metric": "mlogloss",
        "n_jobs": 4
    },

    "CATBOOST_PARAMS": {
        "iterations": 1000,
        "learning_rate": 0.03,
        "depth": 10,
        "random_seed": 42,
        "verbose": 0,
        "thread_count": 4
    }
}

print("⚙️ Configuration Loaded Successfully!")
print(f"   • CV Strategy: {CONFIG['N_SPLITS']}-Fold Stratified")
print(f"   • Active Models: LGBM={CONFIG['USE_LGBM']}, XGB={CONFIG['USE_XGB']}, CatBoost={CONFIG['USE_CATBOOST']}")
print(f"   • Ensemble Weights: {CONFIG['WEIGHTS']}")


⚙️ Configuration Loaded Successfully!
   • CV Strategy: 5-Fold Stratified
   • Active Models: LGBM=True, XGB=True, CatBoost=True
   • Ensemble Weights: {'lgbm': 0.325, 'xgb': 0.575, 'catboost': 0.1}


## 📊 SECTION 4: DATA LOADING & INTEGRITY CHECKS


In [4]:
# ==============================================================================
# 📊 SECTION 4: DATA LOADING & INTEGRITY CHECKS
# ==============================================================================

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print(f"📈 Train Set:  {train_df.shape[0]:,} rows, {train_df.shape[1]} columns")
print(f"📉 Test Set:   {test_df.shape[0]:,} rows, {test_df.shape[1]} columns")
print(f"📋 Sample Sub: {sample_sub.shape[0]:,} rows, {sample_sub.shape[1]} columns")

# --------------------------------------------------------------------------
# Competition schema
# --------------------------------------------------------------------------
TARGET = "Status"
CLASS_ORDER = ["C", "CL", "D"]
SUBMISSION_COLUMNS = ["Status_C", "Status_CL", "Status_D"]

required_train_columns = set(test_df.columns) | {TARGET}
missing_required = required_train_columns - set(train_df.columns)

if missing_required:
    raise ValueError(
        f"Training data is missing expected columns: {sorted(missing_required)}"
    )

if TARGET not in train_df.columns:
    raise ValueError(f"Target column '{TARGET}' was not found in train.csv.")

unknown_classes = sorted(
    set(train_df[TARGET].dropna().astype(str).unique()) - set(CLASS_ORDER)
)

if unknown_classes:
    raise ValueError(f"Unexpected target classes found: {unknown_classes}")

print("\n🎯 Target Distribution:")
counts = train_df[TARGET].value_counts()
pcts = train_df[TARGET].value_counts(normalize=True) * 100

for status in CLASS_ORDER:
    count = int(counts.get(status, 0))
    pct = float(pcts.get(status, 0))
    print(f"   • {status}: {count:,d} ({pct:.2f}%)")

print("\n🔍 Missing Values in Training Data:")
missing = train_df.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False).to_string())

print("\n📋 Submission Columns:")
print(sample_sub.columns.tolist())

expected_submission = ["id"] + SUBMISSION_COLUMNS

if sample_sub.columns.tolist() != expected_submission:
    raise ValueError(
        f"Unexpected sample submission columns.\n"
        f"Expected: {expected_submission}\n"
        f"Found:    {sample_sub.columns.tolist()}"
    )

print("\n✅ Dataset schema verified successfully!")


📈 Train Set:  15,000 rows, 20 columns
📉 Test Set:   10,000 rows, 19 columns
📋 Sample Sub: 10,000 rows, 4 columns

🎯 Target Distribution:
   • C: 10,143 (67.62%)
   • CL: 370 (2.47%)
   • D: 4,487 (29.91%)

🔍 Missing Values in Training Data:
Tryglicerides    8397
Cholesterol      8347
Copper           6654
SGOT             6561
Alk_Phos         6558
Spiders          6556
Drug             6552
Hepatomegaly     6552
Ascites          6544
Platelets         554
Prothrombin        24

📋 Submission Columns:
['id', 'Status_C', 'Status_CL', 'Status_D']

✅ Dataset schema verified successfully!


## 🧪 SECTION 5: DOMAIN-INFORMED FEATURE ENGINEERING

The pipeline creates additional features from patient age, operating time,
temperature, vibration, electrical load, maintenance history, sensor drift,
and missing-value patterns.

Categorical columns are encoded consistently across train and test.


In [5]:
# ==============================================================================
# 🧪 SECTION 5: DOMAIN-INFORMED FEATURE ENGINEERING
# ==============================================================================

def engineer_features(df_train, df_test):
    """
    Feature engineering for the supplied cirrhosis dataset.

    The target column Status is deliberately excluded from the feature matrix.
    Categorical variables are encoded using combined train+test category codes
    so the same mapping is used for both datasets.
    """

    combined = pd.concat(
        [df_train.drop(columns=[TARGET]), df_test],
        axis=0,
        ignore_index=True
    ).copy()

    # --------------------------------------------------------------------------
    # 1. Missing-value indicators
    # --------------------------------------------------------------------------
    original_feature_cols = [
        c for c in combined.columns
        if c != "id"
    ]

    for col in original_feature_cols:
        if combined[col].isna().any():
            combined[f"{col}_isna"] = combined[col].isna().astype(int)

    combined["Total_Missing_Count"] = combined[original_feature_cols].isna().sum(axis=1)

    # --------------------------------------------------------------------------
    # 2. Clinical / numerical interaction features
    # --------------------------------------------------------------------------
    eps = 1e-6

    if {"Bilirubin", "Albumin"}.issubset(combined.columns):
        combined["Bilirubin_Albumin_Ratio"] = (
            combined["Bilirubin"] / (combined["Albumin"] + eps)
        )

    if {"Copper", "Albumin"}.issubset(combined.columns):
        combined["Copper_Albumin_Ratio"] = (
            combined["Copper"] / (combined["Albumin"] + eps)
        )

    if {"SGOT", "Alk_Phos"}.issubset(combined.columns):
        combined["SGOT_AlkPhos_Ratio"] = (
            combined["SGOT"] / (combined["Alk_Phos"] + eps)
        )

    if {"Cholesterol", "Tryglicerides"}.issubset(combined.columns):
        combined["Lipid_Load"] = (
            combined["Cholesterol"] + combined["Tryglicerides"]
        )
        combined["Cholesterol_Triglyceride_Ratio"] = (
            combined["Cholesterol"] / (combined["Tryglicerides"] + eps)
        )

    if {"Platelets", "Prothrombin"}.issubset(combined.columns):
        combined["Platelet_Prothrombin_Ratio"] = (
            combined["Platelets"] / (combined["Prothrombin"] + eps)
        )

    if {"N_Days", "Age"}.issubset(combined.columns):
        combined["Days_Age_Ratio"] = (
            combined["N_Days"] / (combined["Age"] + eps)
        )

    if {"Age", "N_Days"}.issubset(combined.columns):
        combined["Age_Years"] = combined["Age"] / 365.25
        combined["Followup_Years"] = combined["N_Days"] / 365.25

    # --------------------------------------------------------------------------
    # 3. Nonlinear numerical transforms
    # --------------------------------------------------------------------------
    log_candidates = [
        "N_Days", "Bilirubin", "Cholesterol", "Copper",
        "Alk_Phos", "SGOT", "Tryglicerides", "Platelets"
    ]

    for col in log_candidates:
        if col in combined.columns:
            combined[f"log_{col}"] = np.log1p(combined[col].clip(lower=0))

    # --------------------------------------------------------------------------
    # 4. Consistent categorical encoding
    # --------------------------------------------------------------------------
    categorical_cols = [
        "Drug", "Sex", "Ascites", "Hepatomegaly",
        "Spiders", "Edema"
    ]

    for col in categorical_cols:
        if col in combined.columns:
            combined[col] = (
                combined[col]
                .astype("string")
                .fillna("__MISSING__")
                .astype("category")
                .cat.codes
            )

    # Stage is ordinal/numeric in the supplied data but may contain missing values.
    # Keep it numeric if possible.
    if "Stage" in combined.columns:
        combined["Stage"] = pd.to_numeric(
            combined["Stage"], errors="coerce"
        )

    # id is an identifier, not a predictive feature.
    feature_cols = [
        c for c in combined.columns
        if c not in ["id", TARGET]
    ]

    X_train = combined.iloc[:len(df_train)][feature_cols].copy()
    X_test = combined.iloc[len(df_train):][feature_cols].copy()

    return X_train, X_test, feature_cols


X_train, X_test, feature_names = engineer_features(train_df, test_df)

target_map = {"C": 0, "CL": 1, "D": 2}
inv_target_map = {0: "C", 1: "CL", 2: "D"}

y_train = train_df[TARGET].astype(str).map(target_map).astype(int).values

print(f"✨ Engineered Feature Space: {len(feature_names)} columns created")
print(f"📋 First 15 Features: {feature_names[:15]}")
print(f"🎯 Target Encoding: {target_map}")


✨ Engineered Feature Space: 47 columns created
📋 First 15 Features: ['N_Days', 'Drug', 'Age', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema', 'Bilirubin', 'Cholesterol', 'Albumin', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides']
🎯 Target Encoding: {'C': 0, 'CL': 1, 'D': 2}


## 🚀 SECTION 6: STRATIFIED K-FOLD TRAINING & ENSEMBLING


In [6]:
# ==============================================================================
# 🚀 SECTION 6: STRATIFIED K-FOLD TRAINING & ENSEMBLING
# ==============================================================================

start_total_time = time.time()

n_splits = CONFIG["N_SPLITS"]

skf = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=CONFIG["RANDOM_STATE"]
)

n_classes = len(CLASS_ORDER)

oof_preds = {
    "lgbm": np.zeros((len(train_df), n_classes)),
    "xgb": np.zeros((len(train_df), n_classes)),
    "catboost": np.zeros((len(train_df), n_classes))
}

test_preds = {
    "lgbm": np.zeros((len(test_df), n_classes)),
    "xgb": np.zeros((len(test_df), n_classes)),
    "catboost": np.zeros((len(test_df), n_classes))
}

print(f"🚀 Training {n_splits}-Fold Stratified Ensemble Pipeline...\n")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    fold_start = time.time()

    X_tr, y_tr = X_train.iloc[train_idx], y_train[train_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train[val_idx]

    # --------------------------------------------------------------------------
    # 1. LightGBM
    # --------------------------------------------------------------------------
    if CONFIG["USE_LGBM"]:
        lgb_model = LGBMClassifier(**CONFIG["LGBM_PARAMS"])
        lgb_model.fit(X_tr, y_tr)

        oof_preds["lgbm"][val_idx] = lgb_model.predict_proba(X_va)
        test_preds["lgbm"] += lgb_model.predict_proba(X_test) / n_splits

    # --------------------------------------------------------------------------
    # 2. XGBoost
    # --------------------------------------------------------------------------
    if CONFIG["USE_XGB"]:
        xgb_model = XGBClassifier(
            **CONFIG["XGB_PARAMS"],
            num_class=n_classes
        )
        xgb_model.fit(X_tr, y_tr)

        oof_preds["xgb"][val_idx] = xgb_model.predict_proba(X_va)
        test_preds["xgb"] += xgb_model.predict_proba(X_test) / n_splits

    # --------------------------------------------------------------------------
    # 3. CatBoost
    # --------------------------------------------------------------------------
    if CONFIG["USE_CATBOOST"]:
        cb_model = CatBoostClassifier(
            **CONFIG["CATBOOST_PARAMS"],
            loss_function="MultiClass",
            classes_count=n_classes
        )
        cb_model.fit(X_tr, y_tr)

        oof_preds["catboost"][val_idx] = cb_model.predict_proba(X_va)
        test_preds["catboost"] += cb_model.predict_proba(X_test) / n_splits

    print(f"   ⏱️ Fold {fold}/{n_splits} Finished in {time.time() - fold_start:.2f}s")

total_training_time = time.time() - start_total_time
print(f"\n⚡ All {n_splits} folds completed in {total_training_time:.2f}s")


🚀 Training 5-Fold Stratified Ensemble Pipeline...

   ⏱️ Fold 1/5 Finished in 141.99s
   ⏱️ Fold 2/5 Finished in 141.72s
   ⏱️ Fold 3/5 Finished in 142.39s
   ⏱️ Fold 4/5 Finished in 142.09s
   ⏱️ Fold 5/5 Finished in 141.86s

⚡ All 5 folds completed in 710.06s


## 📈 SECTION 7: OOF VALIDATION & PROBABILITY ENSEMBLE

This section combines the trained model probabilities and creates the final
test-set ensemble used for submission.


In [7]:
# ==============================================================================
# 📈 SECTION 7: OOF VALIDATION & PROBABILITY ENSEMBLE
# ==============================================================================

print("=" * 78)
print("📊 OUT-OF-FOLD CROSS-VALIDATION RESULTS")
print("=" * 78)

active_models = [
    name for name, enabled in [
        ("lgbm", CONFIG["USE_LGBM"]),
        ("xgb", CONFIG["USE_XGB"]),
        ("catboost", CONFIG["USE_CATBOOST"])
    ]
    if enabled
]

if not active_models:
    raise ValueError("At least one model must be enabled.")

# --------------------------------------------------------------------------
# Individual model OOF metrics
# --------------------------------------------------------------------------
for model_name in active_models:
    model_pred = np.clip(
        oof_preds[model_name],
        1e-9,
        1.0
    )
    model_pred = model_pred / model_pred.sum(axis=1, keepdims=True)

    model_acc = accuracy_score(
        y_train,
        np.argmax(model_pred, axis=1)
    )

    model_loss = log_loss(
        y_train,
        model_pred,
        labels=list(range(n_classes))
    )

    print(
        f"🔹 {model_name.upper():<10} | "
        f"Accuracy: {model_acc * 100:.2f}% | "
        f"Log Loss: {model_loss:.5f}"
    )

# --------------------------------------------------------------------------
# Normalize user-controlled ensemble weights
# --------------------------------------------------------------------------
raw_weights = CONFIG["WEIGHTS"].copy()

weight_sum = sum(
    max(float(raw_weights.get(model_name, 0.0)), 0.0)
    for model_name in active_models
)

if weight_sum <= 0:
    raise ValueError(
        "The sum of active ensemble weights must be greater than zero."
    )

normalized_weights = {
    model_name:
        max(float(raw_weights.get(model_name, 0.0)), 0.0) / weight_sum
    for model_name in active_models
}

print("\n⚖️ Normalized Ensemble Weights:")
for model_name, weight in normalized_weights.items():
    print(f"   • {model_name}: {weight:.4f}")

# --------------------------------------------------------------------------
# Blend OOF and test probabilities
# --------------------------------------------------------------------------
oof_ensemble = np.zeros((len(train_df), n_classes), dtype=float)
test_ensemble = np.zeros((len(test_df), n_classes), dtype=float)

for model_name in active_models:
    oof_ensemble += normalized_weights[model_name] * oof_preds[model_name]
    test_ensemble += normalized_weights[model_name] * test_preds[model_name]

# Numerical safety + row-wise probability normalization
oof_ensemble = np.clip(oof_ensemble, 1e-9, 1.0)
test_ensemble = np.clip(test_ensemble, 1e-9, 1.0)

oof_ensemble /= oof_ensemble.sum(axis=1, keepdims=True)
test_ensemble /= test_ensemble.sum(axis=1, keepdims=True)

# --------------------------------------------------------------------------
# Final OOF ensemble metrics
# --------------------------------------------------------------------------
ensemble_pred = np.argmax(oof_ensemble, axis=1)

ensemble_accuracy = accuracy_score(
    y_train,
    ensemble_pred
)

ensemble_logloss = log_loss(
    y_train,
    oof_ensemble,
    labels=list(range(n_classes))
)

print("\n" + "=" * 78)
print(
    f"🏆 ENSEMBLE | Accuracy: {ensemble_accuracy * 100:.2f}% "
    f"| Log Loss: {ensemble_logloss:.5f}"
)
print("=" * 78)

print("\n📋 Classification Report:")
print(
    classification_report(
        y_train,
        ensemble_pred,
        labels=list(range(n_classes)),
        target_names=["Status_C", "Status_CL", "Status_D"],
        digits=4,
        zero_division=0
    )
)

print("\n🧩 Confusion Matrix:")
print(
    confusion_matrix(
        y_train,
        ensemble_pred,
        labels=list(range(n_classes))
    )
)

print("\n✅ Test ensemble probabilities generated successfully.")
print(f"   Shape: {test_ensemble.shape}")


📊 OUT-OF-FOLD CROSS-VALIDATION RESULTS
🔹 LGBM       | Accuracy: 85.75% | Log Loss: 0.37510
🔹 XGB        | Accuracy: 85.97% | Log Loss: 0.36519
🔹 CATBOOST   | Accuracy: 85.37% | Log Loss: 0.38620

⚖️ Normalized Ensemble Weights:
   • lgbm: 0.3250
   • xgb: 0.5750
   • catboost: 0.1000

🏆 ENSEMBLE | Accuracy: 85.97% | Log Loss: 0.36340

📋 Classification Report:
              precision    recall  f1-score   support

    Status_C     0.8728    0.9394    0.9048     10143
   Status_CL     0.7231    0.1270    0.2161       370
    Status_D     0.8265    0.7401    0.7810      4487

    accuracy                         0.8597     15000
   macro avg     0.8075    0.6022    0.6340     15000
weighted avg     0.8552    0.8597    0.8508     15000


🧩 Confusion Matrix:
[[9528    9  606]
 [ 232   47   91]
 [1157    9 3321]]

✅ Test ensemble probabilities generated successfully.
   Shape: (10000, 3)


## 📈 SECTION 7: VALIDATION PERFORMANCE & ENSEMBLE BLEND


## 💾 SECTION 8: SUBMISSION GENERATION & INTEGRITY CHECKS


## 💾 SECTION 8: FINAL KAGGLE SUBMISSION GENERATION & VERIFICATION


In [8]:
# ==============================================================================
# 💾 SECTION 8: FINAL KAGGLE SUBMISSION GENERATION & VERIFICATION
# ==============================================================================

submission_df = pd.DataFrame({
    "id": test_df["id"],
    "Status_C": test_ensemble[:, 0],
    "Status_CL": test_ensemble[:, 1],
    "Status_D": test_ensemble[:, 2]
})

# Match the competition sample submission exactly.
submission_df = submission_df[
    ["id", "Status_C", "Status_CL", "Status_D"]
]

submission_df.to_csv(OUTPUT_SUB_PATH, index=False)

row_sums = submission_df[
    ["Status_C", "Status_CL", "Status_D"]
].sum(axis=1)

# --------------------------------------------------------------------------
# Integrity checks
# --------------------------------------------------------------------------
assert len(submission_df) == len(test_df)
assert submission_df["id"].equals(test_df["id"])
assert submission_df[
    ["Status_C", "Status_CL", "Status_D"]
].notnull().all().all()

assert np.isfinite(
    submission_df[
        ["Status_C", "Status_CL", "Status_D"]
    ].to_numpy()
).all()

assert np.allclose(
    row_sums.to_numpy(),
    1.0,
    atol=1e-6
)

assert submission_df.columns.tolist() == [
    "id", "Status_C", "Status_CL", "Status_D"
]

print("=" * 78)
print("💾 FINAL SUBMISSION")
print("=" * 78)
print(f"📁 Saved to: {OUTPUT_SUB_PATH}")
print(f"📏 Shape: {submission_df.shape}")
print(f"🔍 Missing values: {submission_df.isnull().sum().to_dict()}")
print(
    f"🧮 Probability sums: "
    f"min={row_sums.min():.6f}, max={row_sums.max():.6f}"
)

print("\n👀 Submission Preview:")
print(submission_df.head(10).to_string(index=False))

print("\n✅ Submission integrity checks passed!")
print("🚀 Ready to upload submission.csv to Kaggle.")


💾 FINAL SUBMISSION
📁 Saved to: /kaggle/working/submission.csv
📏 Shape: (10000, 4)
🔍 Missing values: {'id': 0, 'Status_C': 0, 'Status_CL': 0, 'Status_D': 0}
🧮 Probability sums: min=1.000000, max=1.000000

👀 Submission Preview:
   id  Status_C  Status_CL  Status_D
15000  0.880538   0.027312  0.092150
15001  0.990953   0.000868  0.008179
15002  0.971459   0.002951  0.025591
15003  0.972136   0.002500  0.025363
15004  0.054064   0.003411  0.942525
15005  0.953669   0.011713  0.034619
15006  0.937548   0.005999  0.056453
15007  0.646348   0.037751  0.315901
15008  0.963952   0.000685  0.035363
15009  0.148007   0.025854  0.826139

✅ Submission integrity checks passed!
🚀 Ready to upload submission.csv to Kaggle.
